In [ ]:
import os
import glob
import pandas as pd
import numpy as np

OUTROOT_FULL = "<PATH_TO_CI_GWAS_OUT>" 
SELECTED_FNAME = "<PATH_TO_CUSKSS_SELECTED_MARKERS_TSV>"

SETUPS = [
    "ldl_pre_post_1to60_no_cvd", 
    "ldl_pre_post_1to60_age5_no_cvd",
]

AGE_LABELS = {
    "pooled": "pooled",
    1: "< 50",
    2: "50 - 55",
    3: "56 - 60",
    4: "60 - 64",
    5: "65+",
}

def _read_setup_selected(outroot, setup, fname):
    pattern = os.path.join(outroot, setup, "**", fname)
    hits = glob.glob(pattern, recursive=True)
    if not hits:
        raise FileNotFoundError(f"No '{fname}' found under: {outroot}/{setup}/**/")
    dfs = []
    for p in sorted(hits):
        d = pd.read_csv(p, sep="\t")
        d["setup"] = setup
        d["path"] = p
        dfs.append(d)
    return pd.concat(dfs, ignore_index=True)

def build_ldl_table(df_full):
    df = _annotate_ldl(df_full)

    tab = (
        df.groupby(["Trait", "Age"], as_index=False)
          .agg(
              n_snps=("is_snp", "sum"),
              n_lof=("is_lof", "sum"),
          )
          .rename(columns={"n_snps": "#SNPs", "n_lof": "#LoF"})
    )

    trait_order = ["LDL pre-treatment", "LDL post-treatment"]
    age_order = ["pooled", "< 50", "50 - 55", "56 - 60", "60 - 64", "65+"]

    tab["Trait"] = pd.Categorical(tab["Trait"], categories=trait_order, ordered=True)
    tab["Age"] = pd.Categorical(tab["Age"], categories=age_order, ordered=True)
    tab = tab.sort_values(["Trait", "Age"]).reset_index(drop=True)
    return tab

    return tab

In [ ]:
df_full = pd.concat([_read_setup_selected(OUTROOT_FULL, s, SELECTED_FNAME) for s in SETUPS], ignore_index=True)

In [ ]:
df_full = df_full[df_full["p_fdr"] < 0.05]

In [ ]:
ldl_table = build_ldl_table(df_full)